# Analisis Dataset Piracy Dataset

Notebook ini dibuat untuk dijalankan secara lokal di **VS Code / Jupyter Notebook**, bukan khusus Google Colab.

Tahapan: Data Loading, Data Cleaning, EDA, Train-Test Split, Data Modeling, evaluasi MAE/MSE/R², visualisasi, dan kesimpulan.

**Sebelum menjalankan:** letakkan file `piracydataset.csv` di folder yang sama dengan notebook ini. Jika library belum terpasang, jalankan di terminal: `python -m pip install pandas numpy matplotlib seaborn scikit-learn`.

## 1. Data Loading

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Mencari CSV di folder kerja saat ini dan folder notebook
csv_path = Path.cwd() / "piracydataset.csv"
if not csv_path.exists():
    # Tampilkan lokasi kerja dan file CSV yang tersedia untuk membantu menemukan file
    csv_files = list(Path.cwd().glob("*.csv"))
    if len(csv_files) == 1:
        csv_path = csv_files[0]
    else:
        raise FileNotFoundError(
            f"File 'piracydataset.csv' tidak ditemukan.\n"
            f"Folder kerja saat ini: {Path.cwd()}\n"
            f"File CSV yang ditemukan: {[p.name for p in csv_files]}\n"
            "Letakkan piracydataset.csv di folder kerja tersebut atau sesuaikan csv_path."
        )

df = pd.read_csv(csv_path)
print("File dibaca:", csv_path.resolve())
print("Ukuran dataset:", df.shape)
print("\\nLima data pertama:")
display(df.head())
print("\\nInformasi dataset:")
df.info()

## 2. Data Cleaning

In [ ]:
print("Data kosong per kolom sebelum cleaning:")
print(df.isna().sum())
print("\\nJumlah baris duplikat penuh:", df.duplicated().sum())

# Buang kolom indeks dan identifier; identifier tidak dipakai sebagai fitur model
df_clean = df.drop(columns=["Unnamed: 0", "discordid", "minecraftid"], errors="ignore").copy()

# Pastikan target age numerik
df_clean["age"] = pd.to_numeric(df_clean["age"], errors="coerce")

# Hapus baris dengan target usia kosong/tidak valid
df_clean = df_clean.dropna(subset=["age"])

# Pastikan kolom fitur tersedia dan isi nilai kategorikal kosong secara eksplisit
required_columns = ["age", "client", "version"]
missing_columns = [col for col in required_columns if col not in df_clean.columns]
if missing_columns:
    raise ValueError(f"Kolom wajib tidak ditemukan: {missing_columns}")

df_clean["client"] = df_clean["client"].fillna("Unknown").astype(str)
df_clean["version"] = df_clean["version"].fillna("Unknown").astype(str)

print("\\nUkuran setelah cleaning:", df_clean.shape)
print("Data kosong setelah cleaning:")
print(df_clean.isna().sum())
display(df_clean.head())

## 3. EDA (Exploratory Data Analysis)

In [ ]:
print("Statistik deskriptif:")
display(df_clean.describe(include="all"))

print("\\nDistribusi client:")
print(df_clean["client"].value_counts())

print("\\nDistribusi version:")
print(df_clean["version"].value_counts())

print("\\nStatistik usia berdasarkan client:")
display(df_clean.groupby("client")["age"].agg(["count", "mean", "median", "min", "max"]))

print("\\nStatistik usia berdasarkan version:")
display(df_clean.groupby("version")["age"].agg(["count", "mean", "median", "min", "max"]))

plt.figure(figsize=(8, 5))
sns.histplot(data=df_clean, x="age", bins=15, kde=True)
plt.title("Distribusi Usia")
plt.xlabel("Usia")
plt.ylabel("Frekuensi")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
sns.countplot(data=df_clean, x="client")
plt.title("Distribusi Client")
plt.xlabel("Client")
plt.ylabel("Jumlah Data")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
sns.countplot(data=df_clean, x="version")
plt.title("Distribusi Version Minecraft")
plt.xlabel("Version")
plt.ylabel("Jumlah Data")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

## 4. Pembagian Dataset Train dan Test

In [ ]:
from sklearn.model_selection import train_test_split

X = df_clean[["client", "version"]]
y = df_clean["age"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Jumlah data training: {len(X_train)} ({len(X_train) / len(df_clean):.1%})")
print(f"Jumlah data testing: {len(X_test)} ({len(X_test) / len(df_clean):.1%})")

## 5. Data Modeling

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

# Encode fitur kategorikal agar dapat diproses model
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), ["client", "version"])
    ]
)

models = {
    "Dummy Baseline": DummyRegressor(strategy="mean"),
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        min_samples_leaf=5
    )
}

print("Model yang akan diuji:")
for name in models:
    print("-", name)

## 6. Evaluasi Model: MAE, MSE, dan R²

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

hasil = []
trained_models = {}

for nama, model in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    trained_models[nama] = pipeline

    hasil.append({
        "Model": nama,
        "MAE": mean_absolute_error(y_test, y_pred),
        "MSE": mean_squared_error(y_test, y_pred),
        "R2": r2_score(y_test, y_pred)
    })

hasil_df = pd.DataFrame(hasil).sort_values("MAE").reset_index(drop=True)
print("Hasil evaluasi (MAE dan MSE lebih kecil lebih baik; R² lebih tinggi lebih baik):")
display(hasil_df.round(4))

hasil_df.to_csv("hasil_evaluasi_model.csv", index=False)
print("Hasil evaluasi disimpan ke:", (Path.cwd() / "hasil_evaluasi_model.csv").resolve())

## 7. Visualisasi Perbandingan Model

In [ ]:
for metric in ["MAE", "MSE", "R2"]:
    plt.figure(figsize=(8, 5))
    sns.barplot(data=hasil_df, x="Model", y=metric)
    plt.title(f"Perbandingan Model berdasarkan {metric}")
    plt.xlabel("Model")
    plt.ylabel(metric)
    plt.xticks(rotation=15, ha="right")
    plt.tight_layout()
    plt.show()

## 8. Kesimpulan Otomatis

In [ ]:
best_mae = hasil_df.loc[hasil_df["MAE"].idxmin()]
baseline = hasil_df[hasil_df["Model"] == "Dummy Baseline"].iloc[0]

print("KESIMPULAN ANALISIS")
print(f"Jumlah data setelah cleaning: {len(df_clean)} baris.")
print(f"Training: {len(X_train)} baris; testing: {len(X_test)} baris.")
print(f"Model dengan MAE terendah: {best_mae['Model']} (MAE={best_mae['MAE']:.4f}).")
if best_mae["Model"] == "Dummy Baseline":
    print("Model machine learning belum mengungguli baseline rata-rata berdasarkan MAE.")
else:
    print("Model dengan MAE terendah mengungguli baseline berdasarkan MAE pada split ini.")
print("MAE adalah rata-rata kesalahan absolut; MSE adalah rata-rata kuadrat kesalahan; R² mengukur kemampuan menjelaskan variasi target.")
print("Catatan: fitur client dan version mungkin tidak cukup informatif untuk memprediksi usia. Hindari menyimpulkan sebab-akibat dari hasil ini.")